# Clasificador de textos en esperanto
**Segunda Guerra Mundial · Guerra Civil Española · Guerras Napoleónicas**

Minería de Textos · CUNEF · 2026/2027

## Qué hace este cuaderno

Dado un párrafo escrito en esperanto, decide de cuál de las tres guerras habla. Se construye en tres fases, de arriba abajo:

| Fase | Pregunta que responde | Qué se obtiene |
|---|---|---|
| **1 · Dataset** | ¿De dónde salen los ejemplos para aprender? | `data/dataset.csv`: 1.803 párrafos etiquetados (601 por clase) |
| **2 · NLP** | ¿Cómo convertimos un texto en palabras analizadas? | `nlp(texto)`: tokenizador, etiquetador gramatical, lematizador y stop-words, hechos desde cero con reglas |
| **3 · Clasificación** | ¿Cómo aprende el modelo y cómo de bien lo hace? | Modelo entrenado, métricas y clasificación de páginas web reales |

## Cómo ejecutarlo

1. Ejecutar todo (*Entorno de ejecución → Ejecutar todas*). Tarda menos de un minuto.
2. La variable `GENERAR_DATASET` de la primera celda elige entre cargar el CSV ya generado (`False`, valor por defecto) o reconstruirlo desde la API de Vikipedio (`True`, varios minutos por las pausas entre peticiones).

## Cómo leer el código

- Todas las librerías y parámetros están en la **primera celda de código**. Los nombres en MAYÚSCULAS son constantes que se pueden cambiar.
- Cada función lleva un *docstring* (el texto entre `"""` bajo su nombre) con **Parámetros**, **Devuelve** y **Cómo funciona**.
- Los comentarios con `#` explican el porqué de cada decisión y qué guarda cada variable.

In [ ]:
# =============================================================================
# IMPORTACIONES Y PARÁMETROS GLOBALES
# =============================================================================
# Todo lo que se puede cambiar para modificar el comportamiento del cuaderno
# está en esta celda. El resto del cuaderno solo LEE estas variables.

# --- Librerías ---
import os                            # crear la carpeta donde se guarda el CSV
import re                            # expresiones regulares: limpieza de texto y reglas del esperanto
import time                          # pausa entre peticiones para no saturar el servidor
import requests                      # peticiones HTTP (API de Vikipedio y páginas web)
import pandas as pd                  # tablas (DataFrame)
from bs4 import BeautifulSoup        # extraer los párrafos de una página HTML

from sklearn.feature_extraction.text import TfidfVectorizer          # texto -> matriz TF-IDF
from sklearn.model_selection import train_test_split                 # partición entrenamiento / test
from sklearn.naive_bayes import MultinomialNB                        # clasificador
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# --- Modo de ejecución ---
# False -> descarga el CSV ya generado desde GitHub (rápido). Valor por defecto.
# True  -> reconstruye el dataset desde la API de Vikipedio (varios minutos).
GENERAR_DATASET = False

# Semilla de todos los procesos aleatorios (equilibrado de clases y partición train/test).
# Misma semilla = mismos resultados.
SEMILLA = 42

# --- Qué datos se piden y cómo se limpian ---
# Clases del problema: nombre de la clase -> categoría raíz en Vikipedio.
# Los nombres de categoría distinguen mayúsculas y minúsculas.
CATEGORIAS = {
    "Segunda Guerra Mundial": "Kategorio:Dua Mondmilito",
    "Guerra Civil Española": "Kategorio:Hispana Enlanda Milito",
    "Guerras Napoleónicas": "Kategorio:Napoleonaj militoj",
}
# De cada clase se toman los artículos de la categoría raíz y los de sus subcategorías directas,
# salvo estas subcategorías, que no hablan de la guerra en sí.
SUBCATS_EXCLUIDAS = {
    "Kategorio:Sekvoj de la Dua Mondmilito",   # consecuencias: es posguerra
    "Kategorio:Frankismo",                     # dictadura de 1939 a 1975: es posguerra
    "Kategorio:Brit-usona milito de 1812",     # es otra guerra
}
# Artículos concretos que se descartan.
ARTICULOS_EXCLUIDOS = {
    "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito",   # es una lista de nombres, no prosa
}
# Un párrafo es una línea de texto con al menos tantas palabras como indica esta constante.
MIN_PALABRAS = 10

# --- Conexión con la web ---
API_URL = "https://eo.wikipedia.org/w/api.php"   # API de MediaWiki de Vikipedio
URL_WIKI = "https://eo.wikipedia.org/wiki/"      # las páginas se piden como URL_WIKI + título con "_" en vez de espacios
# Wikimedia exige que cada programa se identifique con un User-Agent.
HEADERS = {"User-Agent": "ClasificadorMilito/0.1 (https://github.com/Freibott/clasificador-milito; practica CUNEF)"}
PAUSA = 1.0              # segundos de cortesía entre peticiones (evita el error 429, "demasiadas peticiones")

# --- Ficheros ---
RUTA_CSV = "data/dataset.csv"          # dónde se guarda el dataset al generarlo
URL_CSV_REMOTO = "https://raw.githubusercontent.com/Freibott/clasificador-milito/main/data/dataset.csv"
# Lista de stop-words de la asignatura (repositorio del profesor).
URL_STOPWORDS = "https://raw.githubusercontent.com/jparisu/nlp-esperantilo/main/resources/esperanto/vortoj.json"

## Fase 1 · Dataset

**Qué se hace.** Con la API de Vikipedio (Wikipedia en esperanto) se piden los artículos de cada categoría, se descarga su texto plano, se parte en párrafos y se limpia. Cada párrafo hereda como etiqueta la clase de su artículo.

**Decisiones de diseño.** El estudio completo está en `docs/bitacora_exploracion_fase1.md` y `docs/estudio_profundidad_fase1.md`.

| Decisión | Motivo |
|---|---|
| **Tres guerras del mismo tema** | El enunciado pide clases cercanas. Las clases iniciales (WWI, WWII y Guerra Fría) compartían 32 artículos a profundidad 1, lo que mete ruido en las etiquetas. Otras candidatas (Vietnam, Afganistán, Aceh, guerra civil china, Corea) tenían pocos datos. |
| **Profundidad 1** (categoría + subcategorías directas) | Es la mínima con la que las Guerras Napoleónicas superan los 300 párrafos recomendados. La profundidad 2 les aporta solo 11 más y duplica la Segunda Guerra Mundial. |
| **Excluir 3 subcategorías y 1 artículo** | Son de posguerra, de otra guerra o una lista de nombres (ver `SUBCATS_EXCLUIDAS` y `ARTICULOS_EXCLUIDOS`). |
| **Quitar párrafos repetidos** | Un mismo párrafo en dos clases sería un ejemplo con etiquetas contradictorias. |
| **Equilibrar las clases** | Cada clase se recorta al tamaño de la menor, con muestreo aleatorio de semilla fija. Así el *accuracy* no premia a una clase mayoritaria. |

**Ruido de etiquetas.** Heredar la clase del artículo implica que algunos párrafos no hablan de la guerra (bibliografía, genealogía, enlaces de archivo). No se pueden separar a mano, así que se mide su efecto en los errores de la Fase 3.

### Acceso a la API de Vikipedio y descarga de artículos

In [ ]:
def consultar_api(params):
    """Hace una consulta a la API de MediaWiki (Vikipedio) y devuelve la respuesta como diccionario.

    Parámetros
    ----------
    params : dict
        Parámetros de la consulta según la documentación de MediaWiki, por ejemplo
        {"action": "query", "list": "categorymembers", "cmtitle": "Kategorio:Dua Mondmilito"}.

    Devuelve
    --------
    dict con el JSON de la respuesta (response.json() convierte el JSON en diccionarios y listas).

    Cómo funciona
    -------------
    Añade a los parámetros el formato JSON, hace la petición GET con requests y espera PAUSA
    segundos antes de seguir, para no saturar el servidor. Si Vikipedio contesta que hay
    demasiadas peticiones (error 429), basta con esperar un rato y volver a ejecutar.
    """
    params["format"] = "json"
    params["formatversion"] = 2        # formatversion=2 devuelve listas, no diccionarios por ID
    respuesta = requests.get(API_URL, params=params, headers=HEADERS)
    time.sleep(PAUSA)
    return respuesta.json()


def miembros_de_categoria(categoria):
    """Pide a la API el contenido directo de una categoría.

    Parámetros
    ----------
    categoria : str
        Título de la categoría, por ejemplo "Kategorio:Dua Mondmilito".

    Devuelve
    --------
    (paginas, subcats) : dos listas de títulos.
        paginas = artículos de la categoría; subcats = sus subcategorías.

    Cómo funciona
    -------------
    La API devuelve como máximo 500 elementos por petición. Si hay más, la respuesta incluye
    la clave "continue" con los parámetros para pedir la página siguiente. El bucle sigue
    pidiendo hasta que esa clave desaparece. Cada elemento trae su espacio de nombres
    ("ns"): 0 es un artículo y 14 una subcategoría.
    """
    params = {"action": "query", "list": "categorymembers", "cmtitle": categoria,
              "cmlimit": 500, "cmtype": "page|subcat"}
    paginas, subcats = [], []
    while True:
        datos = consultar_api(params)
        for miembro in datos["query"]["categorymembers"]:
            if miembro["ns"] == 0:
                paginas.append(miembro["title"])
            else:
                subcats.append(miembro["title"])
        if "continue" not in datos:        # no hay más páginas de resultados
            return paginas, subcats
        params.update(datos["continue"])   # pide la página siguiente


def articulos_de_clase(raiz):
    """Devuelve los artículos de una clase: los de su categoría raíz y los de sus subcategorías directas.

    Parámetros
    ----------
    raiz : str
        Categoría raíz de la clase, por ejemplo "Kategorio:Dua Mondmilito".

    Devuelve
    --------
    set de títulos de artículos (un set no guarda repetidos).
    """
    articulos, subcats = miembros_de_categoria(raiz)
    for sub in subcats:
        if sub not in SUBCATS_EXCLUIDAS:
            articulos = articulos + miembros_de_categoria(sub)[0]   # [0] = solo los artículos de la subcategoría
    return set(articulos) - ARTICULOS_EXCLUIDOS


def descargar_texto(titulo):
    """Descarga el texto plano de un artículo de Vikipedio.

    Parámetros
    ----------
    titulo : str
        Título exacto del artículo.

    Devuelve
    --------
    str con el texto completo (los títulos de sección aparecen como "== Título =="),
    o "" si el artículo no tiene texto.
    """
    datos = consultar_api({"action": "query", "prop": "extracts", "titles": titulo,
                           "explaintext": 1,             # texto plano, sin HTML
                           "exsectionformat": "wiki",    # títulos de sección como "== Título =="
                           "redirects": 1})              # si el título es una redirección, sigue hasta el artículo
    return datos["query"]["pages"][0].get("extract", "")

### Partir en párrafos y limpiar

Un **párrafo** es una línea del texto plano con `MIN_PALABRAS` palabras o más, sin los títulos de sección (`== ... ==`). Se eliminan las etiquetas HTML, los enlaces y las referencias entre corchetes (`[1]`, pronunciaciones como `[kanAÚ]`) y se cambia `;` por `,` porque `;` es el separador del CSV. Se descartan las citas bibliográficas (ISBN/ISSN o *Autor (año).*), que no son prosa. La misma limpieza se aplica después a las páginas web que se quieren clasificar, para que el modelo vea texto igual al de entrenamiento.

In [ ]:
def parrafo_limpio(linea):
    r"""Limpia una línea de texto y devuelve el párrafo, o None si no vale como párrafo.

    Parámetros
    ----------
    linea : str
        Una línea de texto (de un artículo o de una página web).

    Devuelve
    --------
    str con el párrafo limpio, o None si es una cita bibliográfica o tiene menos de
    MIN_PALABRAS palabras (esto incluye las líneas vacías).

    Cómo funciona
    -------------
    re.sub(patrón, reemplazo, texto) sustituye todo lo que encaja con el patrón:
      <[^>]+>        un "<", uno o más caracteres que no sean ">", y un ">": etiqueta HTML
      https?://\S+   "http://" o "https://" y todo hasta el siguiente espacio: enlace
      \[[^\]]*\]     un "[", cualquier cosa que no sea "]", y un "]": referencia [1]
      \s+            uno o más espacios, tabuladores o saltos de línea: se dejan en un espacio
    Después re.search comprueba si la línea es una cita bibliográfica. El patrón tiene tres
    alternativas separadas por "|" (basta con que se cumpla una):
      ISBN|ISSN             citas de libros y revistas
      \(\d{4}\)[.,:]       un año entre paréntesis seguido de . , o :  -> "Autor (2011)."
      traduko de teksto     aviso de traducción de Vikipedio
    """
    linea = re.sub(r"<[^>]+>", "", linea)         # etiquetas HTML
    linea = re.sub(r"https?://\S+", "", linea)    # enlaces sueltos
    linea = re.sub(r"\[[^\]]*\]", "", linea)      # referencias y pronunciaciones entre corchetes
    linea = linea.replace(";", ",")               # ";" es el separador del CSV
    linea = re.sub(r"\s+", " ", linea).strip()    # espacios repetidos -> uno; quita los de los extremos
    if re.search(r"ISBN|ISSN|\(\d{4}\)[.,:]|traduko de teksto", linea):
        return None
    if len(linea.split()) < MIN_PALABRAS:
        return None
    return linea


def partir_en_parrafos(texto):
    """Parte el texto de un artículo en párrafos limpios.

    Parámetros
    ----------
    texto : str
        Texto plano de un artículo, con saltos de línea y títulos "== Sección ==".

    Devuelve
    --------
    list[str] con los párrafos que superan la limpieza, en el orden del artículo.
    """
    parrafos = []
    for linea in texto.split("\n"):                # cada salto de línea separa un párrafo
        if not linea.strip().startswith("=="):     # los títulos de sección no son párrafos
            parrafo = parrafo_limpio(linea)
            if parrafo is not None:
                parrafos.append(parrafo)
    return parrafos

### Construir el dataset

Se descargan los artículos de cada clase y se parten en párrafos. Después se quitan los párrafos repetidos (dentro de una clase se conserva uno; si el mismo párrafo está en dos clases se quitan todos, porque tendría etiquetas contradictorias), se recorta cada clase al tamaño de la menor y se mezclan las filas. El resultado es un CSV en UTF-8, separador `;`, columnas `text;class`, un párrafo por fila y sin `;` dentro del texto.

In [ ]:
def generar_dataset():
    """Genera el dataset completo desde la API de Vikipedio y lo guarda en RUTA_CSV.

    Devuelve
    --------
    pandas.DataFrame con las columnas "text" y "class", mezclado y con las mismas filas por clase.

    Cómo funciona
    -------------
    1. Para cada clase se obtienen sus artículos, se descarga el texto de cada uno y se parte
       en párrafos. Cada párrafo se guarda junto a la clase del artículo.
    2. drop_duplicates(subset=["text", "class"]) quita los párrafos repetidos dentro de una clase
       (conserva el primero).
    3. drop_duplicates(subset="text", keep=False) quita TODAS las copias de un párrafo que
       aparece en más de una clase: serían ejemplos con etiquetas contradictorias.
    4. Equilibrar: n_min es el tamaño de la clase menor y de cada clase se toman n_min párrafos
       al azar (sample con random_state=SEMILLA hace el azar reproducible).
    5. Se mezclan las filas (sample(frac=1) = todas las filas, en otro orden) y se guarda el CSV.
    """
    filas = []
    for clase, raiz in CATEGORIAS.items():
        for titulo in sorted(articulos_de_clase(raiz)):
            for parrafo in partir_en_parrafos(descargar_texto(titulo)):
                filas.append((parrafo, clase))
    df = pd.DataFrame(filas, columns=["text", "class"])

    df = df.drop_duplicates(subset=["text", "class"])
    df = df.drop_duplicates(subset="text", keep=False)

    n_min = df["class"].value_counts().min()
    df = pd.concat([df[df["class"] == clase].sample(n_min, random_state=SEMILLA) for clase in CATEGORIAS])
    df = df.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

    os.makedirs("data", exist_ok=True)
    df.to_csv(RUTA_CSV, sep=";", index=False, encoding="utf-8")
    return df


if GENERAR_DATASET:
    df = generar_dataset()
else:
    try:
        df = pd.read_csv(URL_CSV_REMOTO, sep=";", encoding="utf-8")   # sep=";" porque es el separador del CSV
    except Exception:
        print("Aviso: no se pudo descargar el CSV de GitHub; se genera desde Vikipedio (tarda unos minutos).")
        df = generar_dataset()

print(df.shape)
print(df["class"].value_counts())
df.head()

## Fase 2 · NLP

### Modelo de NLP para esperanto (basado en reglas)

**Objetivo.** Construir desde cero, sin librerías de NLP, un modelo con una API parecida a la de spaCy: `doc = nlp(texto)` devuelve un `Doc` formado por objetos `Token` con `text`, `lemma_`, `pos_` e `is_stop`.

**Por qué funciona con reglas.** El esperanto es regular: la terminación de una palabra fija su categoría gramatical (`-o` sustantivo, `-a` adjetivo, `-e` adverbio, `-as/-is/-os/-us/-u/-i` verbo), y el plural (`-j`) y el acusativo (`-n`) se añaden siempre igual. El análisis de cada palabra sigue este orden y gana la primera regla que encaja:

1. **Pronombre con acusativo** (`min`, `vin`...).
2. **Nombre propio**: palabra con mayúscula que no abre frase.
3. **Palabras cerradas** (artículo, pronombres, preposiciones, conjunciones, partículas, adverbios, interjecciones y los 45 correlativos `kiu`, `tie`, `iam`...) y **numerales** (`du`, `dudek`). Son listas finitas.
4. **Sustantivos, adjetivos y adverbios**, por `-o`, `-a`, `-e`.
5. **Verbos**, por la terminación.
6. Si nada encaja, categoría `X` (palabra extranjera, abreviatura...).

**Etiquetas** (Universal Dependencies, las mismas que usa spaCy): `NOUN` sustantivo · `PROPN` nombre propio · `VERB` verbo · `ADJ` adjetivo · `ADV` adverbio · `DET` determinante · `PRON` pronombre · `ADP` preposición · `CCONJ`/`SCONJ` conjunción coordinante/subordinante · `PART` partícula · `INTJ` interjección · `NUM` número · `PUNCT` puntuación · `X` desconocido.

**Lema.** Es la forma de diccionario: se quitan `-j` y `-n` y el verbo queda en infinitivo (`-i`), siempre en minúsculas. Así `soldato` y `soldatoj` cuentan como la misma palabra en la Fase 3.

**Stop-words.** Palabras muy frecuentes que no distinguen un tema (`la`, `de`, `kaj`...). Se cargan de `vortoj.json` (repositorio de la asignatura), quedándose con las marcadas como `ignorinda`.

**Limitaciones conocidas.**
- No se tratan los participios (`batalanta`, `mortigita`): se etiquetan como adjetivos y su lema es la propia palabra, no el infinitivo del verbo.
- Un nombre propio que abre frase se etiqueta por su terminación (suele salir como sustantivo).
- Las palabras extranjeras que no siguen la morfología del esperanto quedan como `X`.

In [ ]:
# =============================================================================
# FASE 2 · DATOS DE LA GRAMÁTICA DEL ESPERANTO
# =============================================================================

# --- 1. Palabras cerradas ---
# Son listas finitas. Clave = etiqueta gramatical; valor = palabras separadas por espacios
# (más abajo se pasan a lista con .split()).
PALABRAS_CERRADAS = {
    "DET":   "la ambaŭ",
    "PRON":  "mi vi li ŝi ĝi ni ili oni si ci",
    "ADP":   ("al anstataŭ antaŭ apud da de dum ekde ekster el en inter je kontraŭ krom kun laŭ "
              "malantaŭ malgraŭ per po por post preter pri pro sen sub super sur tra trans ĉe ĉirkaŭ ĝis"),
    "CCONJ": "kaj aŭ sed nek do tamen",
    "SCONJ": "ke se ĉar ol kvankam kvazaŭ",
    "PART":  "ne ĉu jes ja jen ĉi ajn ju des",
    "ADV":   ("ankaŭ ankoraŭ apenaŭ baldaŭ eĉ for hieraŭ hodiaŭ jam morgaŭ nun nur plej pli plu "
              "preskaŭ tre tro tuj tute ĵus nepre mem"),
    "INTJ":  "ha he ho hu nu ve adiaŭ",
}

# Diccionario inverso {palabra: etiqueta}: es lo que consulta analizar().
# Por ejemplo CERRADAS["kaj"] vale "CCONJ".
CERRADAS = {}
for etiqueta, palabras in PALABRAS_CERRADAS.items():
    for palabra in palabras.split():
        CERRADAS[palabra] = etiqueta

# Correlativos: prefijo (ki-, ti-, i-, ĉi-, neni-) + terminación. Cada terminación fija la categoría:
# kiu (cuál), tio (eso), kia (qué clase de), kies (de quién), tie (allí), kiam (cuándo)...
# Los dos bucles generan las 5 x 9 = 45 palabras y las añaden a CERRADAS.
TERMINACIONES_CORRELATIVO = {"u": "PRON", "o": "PRON", "a": "DET", "es": "DET",
                             "e": "ADV", "am": "ADV", "al": "ADV", "el": "ADV", "om": "ADV"}
for prefijo in ["ki", "ti", "i", "ĉi", "neni"]:
    for terminacion, etiqueta in TERMINACIONES_CORRELATIVO.items():
        CERRADAS[prefijo + terminacion] = etiqueta

PRONOMBRES = PALABRAS_CERRADAS["PRON"].split()     # mi, vi, li, ŝi, ĝi, ni, ili, oni, si, ci

# --- 2. Numerales ---
# Los numerales se pegan entre sí: du (2), dudek (20), tricent (300), mil.
# La expresión regular resultante, "^(nul|unu|du|...)+$", encaja con una o más de estas
# palabras seguidas (^ = inicio de la palabra, $ = final, + = "una o más veces").
NUMEROS = "nul unu du tri kvar kvin ses sep ok naŭ dek cent mil".split()
NUMERAL = re.compile("^(" + "|".join(NUMEROS) + ")+$")

# --- 3. Stop-words ---
# vortoj.json es un diccionario con una lista "vortoj" de entradas {"vorto": palabra,
# "ignorinda": True/False, ...}. Se guardan las palabras marcadas como "ignorinda"
# (en esperanto, "que se debe ignorar").
try:
    datos_stopwords = requests.get(URL_STOPWORDS, timeout=15).json()
    STOPWORDS = {entrada["vorto"].lower() for entrada in datos_stopwords["vortoj"] if entrada.get("ignorinda")}
except Exception:
    # Si GitHub no responde se usan las palabras cerradas de arriba (artículo, pronombres, preposiciones...)
    # más el verbo "esti" (ser/estar), que es la stop-word verbal más frecuente.
    print("Aviso: no se pudo descargar vortoj.json; se usan las palabras cerradas como stop-words.")
    STOPWORDS = set(CERRADAS) | {"esti"}

# vortoj.json no incluye los posesivos (mia, via, lia...), que son pronombre + "a",
# ni "ambaŭ", aunque son palabras funcionales: se añaden a mano.
STOPWORDS = STOPWORDS | {pronombre + "a" for pronombre in PRONOMBRES} | {"ambaŭ"}
print("Stop-words cargadas:", len(STOPWORDS))

In [ ]:
def quitar_flexion(palabra):
    """Quita el acusativo (-n) y el plural (-j) de una palabra en minúsculas.

    Parámetros
    ----------
    palabra : str
        Palabra en minúsculas, por ejemplo "urbojn" (las ciudades, como complemento directo).

    Devuelve
    --------
    str con la palabra sin esas terminaciones: "urbojn" -> "urbo", "urbo" -> "urbo".

    Cómo funciona
    -------------
    El acusativo es una -n tras una vocal o -j (-on, -an, -en, -un, -jn) y el plural es una -j
    tras -o, -a o -u. Se exige len > 3 para no recortar palabras cortas ("ŝin", "tuj").
    Se quita primero la -n y después la -j, porque en esperanto van en ese orden (-oj-n).
    """
    if len(palabra) > 3 and palabra.endswith("n") and palabra[-2] in "oaeuj":
        palabra = palabra[:-1]
    if len(palabra) > 3 and palabra.endswith("j") and palabra[-2] in "oau":
        palabra = palabra[:-1]
    return palabra


def analizar(palabra, inicio_frase):
    """Análisis de UNA palabra: categoría gramatical y lema.

    Parámetros
    ----------
    palabra : str
        La palabra tal como aparece en el texto (con sus mayúsculas).
    inicio_frase : bool
        True si es la primera palabra de una frase. Una mayúscula inicial indica nombre propio
        solo cuando NO está al principio de la frase.

    Devuelve
    --------
    (pos, lema) : etiqueta gramatical (str) y lema en minúsculas (str).
        analizar("soldatoj", False) -> ("NOUN", "soldato")
        analizar("batalis", False)  -> ("VERB", "batali")

    Cómo funciona
    -------------
    Se prueban las reglas en el orden de la celda de texto de la Fase 2 y se devuelve el
    resultado de la primera que encaja (por eso el orden importa).
    """
    w = palabra.lower()

    # Regla 1: pronombre con acusativo (min, vin, lin, ŝin...): se quita la -n.
    if w.endswith("n") and w[:-1] in PRONOMBRES:
        return "PRON", w[:-1]

    base = quitar_flexion(w)     # base = la palabra sin -n / -j

    # Regla 2: nombre propio = mayúscula que no abre frase.
    if palabra[0].isupper() and not inicio_frase:
        return "PROPN", base

    # Regla 3: palabras cerradas y numerales. Se mira primero la palabra entera ("kun" no debe
    # quedarse en "ku") y después la base ("kiujn" -> "kiu").
    if w in CERRADAS:
        return CERRADAS[w], w
    if base in CERRADAS:
        return CERRADAS[base], base
    if NUMERAL.match(w):
        return "NUM", w

    # Regla 4: sustantivos, adjetivos y adverbios por su vocal final.
    if len(base) >= 3 and base.endswith("o"):
        return "NOUN", base
    if len(base) >= 3 and base.endswith("a"):
        return "ADJ", base
    if len(base) >= 3 and base.endswith("e"):
        return "ADV", base

    # Regla 5: verbos. El lema es la raíz + "i" (el infinitivo).
    if len(base) >= 4 and base.endswith(("as", "is", "os", "us")):
        return "VERB", base[:-2] + "i"
    if len(base) >= 3 and base.endswith(("u", "i")):
        return "VERB", base[:-1] + "i"

    # Regla 6: nada encaja (palabra extranjera, abreviatura...).
    return "X", base

In [ ]:
# =============================================================================
# FASE 2 · API TIPO spaCy: Token, Doc y nlp()
# =============================================================================

# Tras estos signos empieza una frase nueva: la mayúscula de la palabra siguiente
# NO indica nombre propio.
FIN_FRASE = {".", "!", "?", ":"}


class Token:
    """Una palabra, número o signo del texto con su análisis (equivale a spacy.tokens.Token).

    Atributos
    ---------
    text : str      forma original, tal como aparece en el texto
    pos_ : str      categoría gramatical ("NOUN", "VERB", "PUNCT"...); "X" si no se conoce
    lemma_ : str    forma de diccionario en minúsculas ("soldatoj" -> "soldato")
    is_stop : bool  True si es una stop-word
    """

    def __init__(self, text):
        self.text = text
        self.pos_ = "X"          # se rellenan al analizar el token
        self.lemma_ = ""
        self.is_stop = False


class Doc:
    """Resultado de analizar un texto: una secuencia de Token (equivale a spacy.tokens.Doc).

    Se comporta como una lista: len(doc) da el número de tokens, doc[0] el primero y
    "for token in doc" los recorre.
    """

    def __init__(self, text, tokens):
        self.text = text         # texto original completo
        self.tokens = tokens     # lista de objetos Token

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, i):
        return self.tokens[i]


def tokenizar(texto):
    r"""Parte el texto en tokens: números, palabras y signos de puntuación.

    Cómo funciona
    -------------
    re.findall devuelve todas las coincidencias del patrón. El patrón tiene tres alternativas
    separadas por "|" que se prueban en orden:
      \d+          una o más cifras: un número (1939)
      [^\W\d_]+    una o más letras (cualquier carácter que no sea no-alfanumérico, ni dígito, ni "_"),
                    así se incluyen ĉ, ĝ, ĥ, ĵ, ŝ, ŭ
      [^\w\s]      cualquier otro signo: puntuación, comillas...
    tokenizar("En 1939, la milito") -> ['En', '1939', ',', 'la', 'milito']
    """
    return re.findall(r"\d+|[^\W\d_]+|[^\w\s]", texto)


def nlp(texto):
    """Analiza un texto: tokeniza, etiqueta, lematiza y marca las stop-words.

    Parámetros
    ----------
    texto : str
        Texto en esperanto.

    Devuelve
    --------
    Doc con un Token por cada palabra, número o signo.

    Cómo funciona
    -------------
    Se recorren los tokens en orden con la variable inicio_frase (True al empezar y tras
    cada signo de FIN_FRASE). Según cómo es cada token:
      - número: su categoría es NUM y su lema es el propio número.
      - palabra: se llama a analizar(); es stop-word si lo es la forma escrita o su lema
                 (así "estis" cuenta como stop-word porque "esti" lo es).
      - otro:    puntuación.
    """
    tokens = []
    inicio_frase = True
    for forma in tokenizar(texto):
        token = Token(forma)
        if forma.isdigit():
            token.pos_ = "NUM"
            token.lemma_ = forma
            inicio_frase = False
        elif forma.isalpha():
            token.pos_, token.lemma_ = analizar(forma, inicio_frase)
            token.is_stop = forma.lower() in STOPWORDS or token.lemma_ in STOPWORDS
            inicio_frase = False
        else:
            token.pos_ = "PUNCT"
            token.lemma_ = forma
            if forma in FIN_FRASE:
                inicio_frase = True
        tokens.append(token)
    return Doc(texto, tokens)

### Ejemplo del modelo

Análisis de una frase: cada `Token` tiene `text`, `pos_`, `lemma_` e `is_stop`, igual que en spaCy.

In [ ]:
# Ejemplo visual: un token por línea.
for token in nlp("La soldatoj de Napoleono batalis kontraŭ la hispanaj armeoj en 1808."):
    print(f"{token.text:12} {token.pos_:6} {token.lemma_:12} stop={token.is_stop}")

### El modelo aplicado a todo el corpus

Se analizan los 1.803 párrafos. Dos comprobaciones de sentido: qué proporción de tokens queda sin categoría (`X`) y cuáles son los lemas de contenido más frecuentes de cada clase.

In [ ]:
docs = [nlp(texto) for texto in df["text"]]      # lista de Doc: uno por párrafo, en el mismo orden que df

# Categoría gramatical de todos los tokens del corpus.
categorias = pd.Series([token.pos_ for doc in docs for token in doc])
print(f"{len(docs)} párrafos · {len(categorias)} tokens")
print(categorias.value_counts())
print(f"Sin categoría (X): {(categorias == 'X').mean():.1%}")

# Los 8 lemas de contenido (sustantivos, nombres propios, verbos y adjetivos que no son stop-words)
# más frecuentes de cada clase: si el modelo funciona, deben tener sentido temático.
for clase in CATEGORIAS:
    lemas = pd.Series([token.lemma_ for doc, c in zip(docs, df["class"]) if c == clase for token in doc
                       if token.pos_ in ["NOUN", "PROPN", "VERB", "ADJ"] and not token.is_stop])
    print(clase, "->", list(lemas.value_counts().head(8).index))

## Fase 3 · Clasificación

### Representación tabular y entrenamiento

**Representación.** Un modelo de aprendizaje automático necesita números, no texto. Cada párrafo pasa por el modelo de la Fase 2 y se queda con los lemas de sus palabras de contenido (sustantivos, nombres propios, verbos y adjetivos que no son stop-words). Con esos lemas se construye una matriz **TF-IDF**: una fila por párrafo y una columna por lema, descartando los lemas que aparecen en menos de 2 párrafos. El lema evita que `soldato` y `soldatoj` cuenten como palabras distintas, y TF-IDF resta peso a las palabras que aparecen en todas las clases.

**Ojo con la fórmula.** Las diapositivas calculan el IDF por *clases* y sin normalizar para explicar la idea. `TfidfVectorizer` usa el IDF por *textos*, con suavizado, y normaliza cada fila. Es la variante estándar de la librería; la idea (dar más peso a lo que distingue una clase) es la misma.

**Metodología.**
1. Partición 80 % / 20 % estratificada (mantiene la proporción de clases) y con semilla fija. El 20 % (test) no se toca hasta la evaluación final.
2. El vectorizador se ajusta (`fit`) **solo con el entrenamiento**; el test solo se transforma (`transform`). Si se ajustara con todo el dataset, el modelo "vería" el vocabulario del test.
3. Se entrena un clasificador Naive Bayes.
4. Evaluación en test: accuracy, F1 macro y matriz de confusión.

In [ ]:
# Categorías gramaticales que aportan significado temático.
CONTENIDO = ["NOUN", "PROPN", "VERB", "ADJ"]


def a_texto(doc):
    """Convierte un Doc en un texto de lemas, que es lo que recibe el vectorizador.

    Parámetros
    ----------
    doc : Doc
        Resultado de nlp(texto).

    Devuelve
    --------
    str con los lemas separados por espacios, por ejemplo "flago ruĝa nigra vertikala strio".

    Cómo funciona
    -------------
    Se queda con los tokens cuya categoría está en CONTENIDO y que no son stop-words.
    " ".join(lista) une los lemas con espacios. Los números (años, fechas) no se incluyen.
    """
    lemas = [token.lemma_ for token in doc if token.pos_ in CONTENIDO and not token.is_stop]
    return " ".join(lemas)


df["lemas"] = [a_texto(doc) for doc in docs]     # una fila por párrafo: sus lemas de contenido

# Partición 80 % entrenamiento / 20 % test. stratify=... mantiene la proporción de clases en ambos.
# random_state=SEMILLA hace que la partición sea siempre la misma.
X_train, X_test, y_train, y_test = train_test_split(
    df["lemas"], df["class"], test_size=0.2, stratify=df["class"], random_state=SEMILLA)
print(f"Entrenamiento: {len(X_train)} párrafos · test: {len(X_test)} párrafos")
print("Párrafo original:", df.loc[0, "text"][:150])
print("Lemas           :", df.loc[0, "lemas"][:150])

# TF-IDF. min_df=2: se ignoran los lemas que aparecen en menos de 2 párrafos.
vectorizador = TfidfVectorizer(min_df=2)
M_train = vectorizador.fit_transform(X_train)    # fit: aprende el vocabulario y los IDF; transform: crea la matriz
M_test = vectorizador.transform(X_test)          # el test solo se transforma, con el vocabulario del entrenamiento
print("Matriz de entrenamiento:", M_train.shape, "(párrafos x lemas)")

### Modelo y evaluación en test

Se entrena Naive Bayes con el conjunto de entrenamiento y se evalúa una sola vez con el test. La matriz de confusión tiene como filas la clase real y como columnas la clase predicha: la diagonal son los aciertos.

In [ ]:
modelo = MultinomialNB()     # Naive Bayes para recuentos/pesos de palabras; con parámetros por defecto rinde igual que ajustándolos
modelo.fit(M_train, y_train)
prediccion = modelo.predict(M_test)

print(f"Accuracy = {accuracy_score(y_test, prediccion):.3f}"
      f" · F1 macro = {f1_score(y_test, prediccion, average='macro'):.3f}\n")
clases = modelo.classes_
pd.DataFrame(confusion_matrix(y_test, prediccion, labels=clases), index=clases, columns=clases)

### Clasificación de páginas web reales

Para probar el modelo fuera del dataset se descargan páginas de Vikipedio que **no** están en él, se extrae su texto con *web scraping* y se clasifica cada párrafo. La probabilidad de cada clase para la página es la media de las probabilidades de sus párrafos (`predict_proba`), y la clase de la página es la de mayor probabilidad. La clase esperada solo sirve para valorar el resultado: el modelo no la ve. Si una página no se puede descargar, se omite con un aviso.

In [ ]:
# Páginas de prueba: título -> clase esperada.
PAGINAS_WEB = {
    "Batalo de Stalingrado": "Segunda Guerra Mundial",
    "Atako kontraŭ Pearl Harbor": "Segunda Guerra Mundial",
    "Francisko Franko": "Guerra Civil Española",
    "Batalo de Waterloo": "Guerras Napoleónicas",
    "Batalo de Trafalgar": "Guerras Napoleónicas",
    "Napoleono Bonaparte": "Guerras Napoleónicas",
}


def clasificar_pagina(html):
    """Clasifica una página web a partir de su HTML.

    Parámetros
    ----------
    html : str
        Código HTML de la página.

    Devuelve
    --------
    (n_parrafos, probabilidades) : cuántos párrafos tiene la página y una Series con la
    probabilidad de cada clase. Si la página no tiene párrafos, (0, None).

    Cómo funciona
    -------------
    1. BeautifulSoup(html, "html.parser") convierte el HTML en un árbol de etiquetas y
       find_all("p") devuelve todas las etiquetas <p> (los párrafos); .text da su texto sin etiquetas.
    2. Cada párrafo recibe la MISMA limpieza y el mismo preprocesado que en el entrenamiento:
       parrafo_limpio -> nlp -> a_texto -> vectorizador.transform -> modelo.predict.
    3. predict_proba da, para cada párrafo, la probabilidad de cada clase. La de la página es
       la media de las de sus párrafos (mean(axis=0) = media de cada columna).
    """
    sopa = BeautifulSoup(html, "html.parser")
    parrafos = [parrafo_limpio(p.text) for p in sopa.find_all("p")]
    parrafos = [p for p in parrafos if p is not None]
    if len(parrafos) == 0:
        return 0, None
    matriz = vectorizador.transform([a_texto(nlp(p)) for p in parrafos])
    probabilidades = pd.Series(modelo.predict_proba(matriz).mean(axis=0), index=modelo.classes_)
    return len(parrafos), probabilidades


filas = []
for titulo, esperada in PAGINAS_WEB.items():
    try:
        respuesta = requests.get(URL_WIKI + titulo.replace(" ", "_"), headers=HEADERS, timeout=15)
        respuesta.raise_for_status()       # lanza un error si el código de respuesta no es 200
    except requests.RequestException:
        print(f"Aviso: no se pudo descargar «{titulo}»; se omite.")
        continue
    n_parrafos, probabilidades = clasificar_pagina(respuesta.text)
    if n_parrafos == 0:
        print(f"Aviso: «{titulo}» no tiene párrafos válidos; se omite.")
        continue
    predicha = probabilidades.idxmax()     # la clase con mayor probabilidad
    fila = {"página": titulo, "párrafos": n_parrafos, "clase esperada": esperada, "clase predicha": predicha}
    fila.update(probabilidades.round(2).add_prefix("P: ").to_dict())     # una columna por clase: "P: Guerras Napoleónicas"...
    fila["acierto"] = "✔" if predicha == esperada else "✘"
    filas.append(fila)
pd.DataFrame(filas)

### Resultados y conclusiones

**Fase 2 · modelo de NLP.** El ejemplo de la frase muestra el análisis de cada token (`pos_`, `lemma_`, `is_stop`). Sobre los 1.803 párrafos del dataset, el 2,6 % de los tokens se queda sin categoría (`X`): palabras extranjeras, abreviaturas y participios con terminaciones no previstas. Los lemas más frecuentes de cada clase tienen sentido temático (`hispana`, `napoleono`, `germana`...).

**Fase 3 · evaluación en test (361 párrafos).** Accuracy 0,853 y F1 macro 0,853. Los errores se reparten entre las tres clases (de 2 a 12 por celda fuera de la diagonal); la confusión menos frecuente es Guerra Civil Española → Guerras Napoleónicas (2). Con `GaussianNB` y el mismo vectorizador el accuracy baja a 0,828 y con una SVM lineal a 0,823, por eso se usa Naive Bayes.

**Ruido de etiquetas.** Parte de los errores son párrafos que no hablan de la guerra (bibliografía, genealogía, enlaces de archivo) y heredan la etiqueta de su artículo. No se ha medido cuántos.

**Páginas web.** Al ejecutar la última celda se obtiene, para cada página, la probabilidad de cada clase (media de sus párrafos) y la clase predicha. Los resultados dependen de la descarga en directo, así que se comentan con la tabla delante.

**Limitaciones.**
- Las páginas de prueba proceden de la misma fuente que el entrenamiento (Vikipedio). No se ha medido el comportamiento con otros estilos de web.
- La clase esperada de cada página la fijamos nosotros según su tema; la de Francisko Franko es discutible porque su biografía cubre también la dictadura.
- Las páginas con pocos párrafos de 10 palabras o más dan una probabilidad poco fiable.